# Role 2: Modeling and Multi-Modal Fusion - Process Summary\n
\n
This notebook documents the tasks completed for Role 2 in the Bearing RUL Prediction project.\n
\n
## Objectives\n
- Build and test RUL prediction models from simple baselines to the proposed multi-modal fusion model.\n
- Ensure models are configurable, reusable, and meet the input/output contract.\n
- Provide unit tests and auxiliary outputs for analysis.\n
\n
## Key Steps Completed\n
\n
1. **Model Configuration and Factory**\n
   - Defined `ModelConfig` dataclass with all hyperparameters.\n
   - Created model registry/factory (`factory.py`) to instantiate models by key.\n
   - Added support for `return_aux` flag to optionally return attention/scale/temporal weights.\n
\n
2. **Encoder Implementations**\n
   - **Vibration Encoder** (`vibration_encoder.py`): Multi-scale CNN with adaptive pooling, returning early/mid/late features.\n
   - **Temperature Encoder** (`temperature_encoder.py`): Supports feature vector and history representations with MLP/Conv1d/Temporal MLP options.\n
\n
3. **Baseline Models** (`baselines.py`)**, as required:**\n
   - `VibrationCNNRegressor` (vibration only)\n
   - `TemperatureRegressor` (temperature only)\n
   - `LateConcatFusionRegressor` (late fusion of modalities)\n
   - `CNNTemporalRegressor` (concatenate modalities then temporal LSTM)\n
   - Plus three ablation models to isolate contributions:\n
     * `SingleScaleCrossAttentionRegressor`\n
     * `MultiScaleFusionRegressor`\n
     * `MultiScaleCrossAttentionRegressor`\n
\n
4. **Core Components**\n
   - **Cross-Modal Attention** (`cross_attention.py`): Performs cross-attention in both directions, returns attention weights if `return_aux=True`.\n
   - **Scale Fusion** (`scale_fusion.py`): Implements `concat_project` and `weighted_sum` fusion, returns scale weights if `return_aux=True`.\n
   - **Temporal Encoder** (`temporal.py`): GRU/LSTM encoder and dual-path encoder with gated fusion, returns temporal weights if `return_aux=True`.\n
   - **Regression Head** (`regression_head.py`): Two-layer MLP with Softplus for non‑negative output.\n
\n
5. **Full Model** (`multimodal_rul.py`)**:\n
   - Vibration & Temperature Encoders → Cross‑Modal Attention → Scale Fusion → Dual‑Path Temporal Encoder → Regression Head → RUL prediction.\n
   - Collects and returns auxiliary outputs (attention, scale, temporal weights) when requested.\n
\n
6. **Testing**\n
   - Unit tests for all modules (`tests/models/`).\n
   - Verified shape, gradient flow, masking, NaN/Inf, deterministic eval, save/load, CPU/GPU parity.\n
   - Over‑fit‑one‑batch smoke test passes.\n
\n
7. **Verification of Requirements**\n
   - All checkboxes in `ROLE_2_MODELING_TASK_PLAN.md` have been marked as done.\n
   - A detailed report is available at `docs/REPORT_ROLE_2_MODELING.md`.\n
\n
## Example Usage\n
The following code snippet shows how to instantiate the full model and run a forward pass with a dummy batch.

## Task Completion Checklist

| Task | Completed Criteria |
|------|--------------------|
| 1. Model API, Base Classes, Configuration | [x] ModelConfig dataclass, factory, return_aux, validation |
| 2. Vibration Encoder | [x] Multi-scale CNN, adaptive pooling, shape [B,T,D_scale], gradient flow, parameter count |
| 3. Temperature Encoder | [x] Feature vector & history, MLP/Conv1d, missing mask handled, output shape |
| 4. Baseline Models | [x] Four baselines + three ablation models, each forward [B,1], config toggles, parameter count |
| 5. Cross-Modal Attention Block | [x] Cross-attention in both directions, proper masking, residual, norm, dropout, batch independence |
| 6. Multi-Scale Fusion | [x] concat_project & weighted_sum, scale weights sum to 1, gradient, aux output |
| 7. Temporal Modeling | [x] GRU/LSTM, dual-path with gated fusion, sequence length variable, temporal weights aux |
| 8. Regression Head & Full Model | [x] Non-negative output via Softplus, full model backward, save/load, output shape [B,1] |
| 9. Model Tests & Integration Readiness | [x] Shape, gradient, mask, NaN/Inf, deterministic eval, save/load, CPU/GPU parity, overfit-one-batch |
| Definition of Done | [x] All criteria satisfied: four baselines + full model, shared API, support for vibration lengths 1024/2560, temperature representations, cross-attention no batch mixing, multi-scale projection, distinct fast/slow paths, RUL shape & non-negativity, all tests pass, synthetic & real batch compatibility, Role 3 can instantiate models, no placeholder metrics |


In [ ]:
# Import necessary libraries\n
import torch\n
from src.models.factory import build_model\n
from src.models.config import ModelConfig\n
\n
# Create a configuration (small batch for demo)\n
cfg = ModelConfig(batch_size=2, sequence_length=16, device='cpu')\n
\n
# Build the full proposed model\n
model = build_model('full_dual_path', cfg)\n
print(f"Model built: {model.__class__.__name__}")\n
\n
# Create a dummy batch matching the contract\n
batch = {\n
    'vibration': torch.randn(2, 16, 2, 1024),  # [B, T, 2, L]\n
    'temperature': torch.randn(2, 16, 7),       # [B, T, F_temp] (feature vector)\n
    'temperature_mask': torch.ones(2, 16, 7, dtype=torch.bool),\n
    'rul': torch.rand(2, 1) * 1000               # [B, 1] (not used in forward)\n
}\n
\n
# Set model to evaluation mode and run forward\n
model.eval()\n
with torch.no_grad():\n
    output = model(batch)\n
    print('RUL prediction shape:', output['rul_pred'].shape)\n
    print('RUL prediction values:\n', output['rul_pred'])\n
    if cfg.return_aux:\n
        print('Auxiliary keys:', list(output['aux'].keys()))\n
        # Example: show shape of scale weights if present\n
        if 'scale_weights' in output['aux']:\n
            print('Scale weights shape:', output['aux']['scale_weights'].shape)\n